# LightGCN Implementation for Recommendation

https://medium.com/@jn2279/better-recommender-systems-with-lightgcn-a0e764af14f9

## 1. Imports

In [29]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import degree
from tqdm.notebook import tqdm
import random
import os
from sklearn.metrics import roc_auc_score

from src.data_loaders import load_movielens_data, load_steam_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


## 2. Data Loading and Preparation

In [30]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

ratings_df = ratings_df[ratings_df['rating'] >= 4.0]
movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df['movieId'].astype(str)
})

steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str)
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

unique_users = all_interactions['user_id'].unique()
unique_items = all_interactions['item_id'].unique()

user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}

num_users = len(user_map)
num_items = len(item_map)

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

all_interactions['user_idx'] = all_interactions['user_id'].map(user_map)
all_interactions['item_idx'] = all_interactions['item_id'].map(item_map)

user_indices = torch.LongTensor(all_interactions['user_idx'].values)
item_indices = torch.LongTensor(all_interactions['item_idx'].values)

edge_index = torch.stack([
    torch.cat([user_indices, item_indices + num_users]),
    torch.cat([item_indices + num_users, user_indices])
], dim=0)

print("Edge index created:")
print(edge_index.shape)


Total unique interactions: 12497401
Number of users: 184419
Number of items: 43660
Edge index created:
torch.Size([2, 24994802])


## 2.1 Data Splitting for Evaluation

In [31]:
def split_data(interactions, test_size=0.2):
    test_indices = np.random.choice(interactions.index, size=int(len(interactions) * test_size), replace=False)
    test = interactions.loc[test_indices]
    train = interactions.drop(test_indices)
    return train, test

train_interactions, test_interactions = split_data(all_interactions)

user_indices_train = torch.LongTensor(train_interactions['user_idx'].values)
item_indices_train = torch.LongTensor(train_interactions['item_idx'].values)

edge_index_train = torch.stack([
    torch.cat([user_indices_train, item_indices_train + num_users]),
    torch.cat([item_indices_train + num_users, user_indices_train])
], dim=0)


## 3. LightGCN Model Definition

In [32]:
from src.models.lightgcn import LightGCNModel as LightGCN

## 4. Training Setup

In [33]:
from src.models.lightgcn import BPRDataset, bpr_loss

embedding_dim = 32
num_layers = 3
batch_size = 4096
learning_rate = 1e-3
epochs = 2

train_dataset = BPRDataset(train_interactions, num_items, user_map)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = LightGCN(num_users, num_items, embedding_dim, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
edge_index_train = edge_index_train.to(device)


## 5. Training Loop

In [36]:
model_path = 'models/lightgcn_model.pth'

if os.path.isfile(model_path):
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    print(f"Model loaded from {model_path}")
else:
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for user_batch, pos_item_batch, neg_item_batch in progress_bar:
            optimizer.zero_grad()
            
            user_final_embs, item_final_embs = model(edge_index_train)
            
            user_embs = user_final_embs[user_batch.to(device)]
            pos_item_embs = item_final_embs[pos_item_batch.to(device)]
            neg_item_embs = item_final_embs[neg_item_batch.to(device)]
            
            user_embs_0 = model.user_embedding(user_batch.to(device))
            pos_item_embs_0 = model.item_embedding(pos_item_batch.to(device))
            neg_item_embs_0 = model.item_embedding(neg_item_batch.to(device))
            
            loss = bpr_loss(user_embs, pos_item_embs, neg_item_embs,  
                            user_embs_0, pos_item_embs_0, neg_item_embs_0)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
            
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


Model loaded from models/lightgcn_model.pth


## 6. Making Recommendations

In [37]:
def get_recommendations(user_id_str, model, top_k=10):
    model.eval()
    
    if user_id_str not in user_map:
        print(f"User '{user_id_str}' not found.")
        return
    user_idx = user_map[user_id_str]
    
    with torch.no_grad():
        user_final_embs, item_final_embs = model(edge_index_train)
        
        user_emb = user_final_embs[user_idx]
        
        scores = torch.matmul(user_emb, item_final_embs.T)
        
        top_k_scores, top_k_indices = torch.topk(scores, k=top_k)
        
        inv_item_map = {i: item for item, i in item_map.items()}
        
        print(f"Top {top_k} recommendations for user '{user_id_str}':")
        for i, score in zip(top_k_indices.cpu().numpy(), top_k_scores.cpu().numpy()):
            if 'steam' in inv_item_map[i]:
                print(f"  - [steam]Item: {inv_item_map[i]}, Title: {items_df_steam.loc[items_df_steam['app_id'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")
            else:
                print(f"  - [movie]Item: {inv_item_map[i]}, Title: {movies_df.loc[movies_df['movieId'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")

        print(f"Least preferred {top_k} items for user '{user_id_str}':")
        bottom_k_scores, bottom_k_indices = torch.topk(scores, k=top_k, largest=False)
        for i, score in zip(bottom_k_indices.cpu().numpy(), bottom_k_scores.cpu().numpy()):
            if 'steam' in inv_item_map[i]:
                print(f"  - [steam]Item: {inv_item_map[i]}, Title: {items_df_steam.loc[items_df_steam['app_id'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")
            else:
                print(f"  - [movie]Item: {inv_item_map[i]}, Title: {movies_df.loc[movies_df['movieId'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")

sample_user_id = 'movielens_user_2'
print(f"Items liked by the user ({sample_user_id}):")

liked_items = all_interactions[all_interactions['user_id'] == sample_user_id]['item_id']
for item in liked_items:
    print(f"  - {item}, Title: {movies_df.loc[movies_df['movieId'] == int(item.split('_')[-1]), 'title'].values[0]}")

get_recommendations(sample_user_id, model)


Items liked by the user (movielens_user_2):
  - movielens_item_110, Title: Braveheart (1995)
  - movielens_item_150, Title: Apollo 13 (1995)
  - movielens_item_151, Title: Rob Roy (1995)
  - movielens_item_236, Title: French Kiss (1995)
  - movielens_item_260, Title: Star Wars: Episode IV - A New Hope (1977)
  - movielens_item_318, Title: Shawshank Redemption, The (1994)
  - movielens_item_333, Title: Tommy Boy (1995)
  - movielens_item_349, Title: Clear and Present Danger (1994)
  - movielens_item_356, Title: Forrest Gump (1994)
  - movielens_item_364, Title: Lion King, The (1994)
  - movielens_item_457, Title: Fugitive, The (1993)
  - movielens_item_497, Title: Much Ado About Nothing (1993)
  - movielens_item_527, Title: Schindler's List (1993)
  - movielens_item_534, Title: Shadowlands (1993)
  - movielens_item_589, Title: Terminator 2: Judgment Day (1991)
  - movielens_item_733, Title: Rock, The (1996)
  - movielens_item_914, Title: My Fair Lady (1964)
  - movielens_item_953, Title

## 7. Evaluation

In [38]:
def evaluate(model, test_interactions, train_interactions, k=20):

    model.eval()
    
    with torch.no_grad():
        user_final_embs, item_final_embs = model(edge_index_train)

    test_user_indices = test_interactions['user_idx'].unique()
    
    test_ground_truth = test_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_ground_truth = train_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()

    recalls = []
    precisions = []
    f1_scores = []

    for user_idx in tqdm(test_user_indices, desc='Evaluating'):
        
        ground_truth_items = test_ground_truth.get(user_idx, [])
        if not ground_truth_items:
            continue

        excluded_items = train_ground_truth.get(user_idx, [])
        
        user_emb = user_final_embs[user_idx]

        scores = torch.matmul(user_emb, item_final_embs.T)

        scores[excluded_items] = -np.inf

        _, top_k_indices = torch.topk(scores, k=k)
        top_k_indices = top_k_indices.cpu().numpy()

        hits = np.isin(top_k_indices, ground_truth_items)
        num_hits = np.sum(hits)

        recall = num_hits / len(ground_truth_items)
        precision = num_hits / k
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        recalls.append(recall)
        precisions.append(precision)
        f1_scores.append(f1)

    avg_recall = np.mean(recalls)
    avg_precision = np.mean(precisions)
    avg_f1 = np.mean(f1_scores)

    print(f'Recall@{k}: {avg_recall:.4f}')
    print(f'Precision@{k}: {avg_precision:.4f}')
    print(f'F1-score@{k}: {avg_f1:.4f}')

    return avg_recall, avg_precision, avg_f1

evaluate(model, test_interactions, train_interactions, k=20)


Evaluating:   0%|          | 0/168181 [00:00<?, ?it/s]

Recall@20: 0.1564
Precision@20: 0.0783
F1-score@20: 0.0813


(np.float64(0.15640817974693189),
 np.float64(0.0782719213228605),
 np.float64(0.08127879925060412))

## 7.1 Link Prediction Evaluation (AUC)

In [ ]:
def get_all_user_positive_items(interactions_df):
    user_pos_items = interactions_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
    return user_pos_items

def evaluate_auc(model, test_interactions, all_interactions_for_sampling, num_items, num_neg_samples=100):
    model.eval()
    
    with torch.no_grad():
        user_final_embs, item_final_embs = model(edge_index_train) 
    
    auc_scores = []
    unique_test_users = test_interactions['user_idx'].unique()

    user_pos_items_all = get_all_user_positive_items(all_interactions_for_sampling)

    for user_idx in tqdm(unique_test_users, desc='Evaluating AUC'):
        pos_items_test = test_interactions[test_interactions['user_idx'] == user_idx]['item_idx'].values
        
        if len(pos_items_test) == 0:
            continue

        # Items user has interacted with in any set (train + test)
        interacted_items = user_pos_items_all.get(user_idx, set())
        
        # Sample negative items
        neg_items_sampled = []
        # Ensure we sample enough unique negative items. Max 100 * number of positive items.
        # If num_items is small and user has many positive items, it can be hard to find negatives.
        # Let's make sure we don't loop infinitely if all items are interacted with.
        available_neg_items = list(set(range(num_items)) - interacted_items)
        if not available_neg_items:
            continue # No negative items to sample

        num_neg_to_sample = min(len(pos_items_test) * num_neg_samples, len(available_neg_items))
        neg_items_sampled = random.sample(available_neg_items, num_neg_to_sample)

        if not neg_items_sampled:
            continue

        # Calculate scores for positive and negative items
        user_emb = user_final_embs[user_idx]
        pos_scores = torch.sum(user_emb * item_final_embs[pos_items_test].to(device), dim=1)
        neg_scores = torch.sum(user_emb * item_final_embs[neg_items_sampled].to(device), dim=1)
        
        # Combine scores and labels
        all_scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
        labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
        
        if len(np.unique(labels)) < 2: # roc_auc_score requires both positive and negative classes
            continue

        try:
            auc = roc_auc_score(labels, all_scores)
            auc_scores.append(auc)
        except ValueError:
            # Handle cases where roc_auc_score might fail (e.g., all predictions are the same)
            pass

    avg_auc = np.mean(auc_scores) if auc_scores else 0
    print(f'Average AUC: {avg_auc:.4f}')
    return avg_auc


In [40]:
# Evaluate AUC
avg_auc = evaluate_auc(model, test_interactions, all_interactions, num_items, num_neg_samples=100)


Evaluating AUC:   0%|          | 0/168181 [00:00<?, ?it/s]

Average AUC: 0.9783
